# EXTRA EXERCISE 1

We are investigating the effect of the influence of the heat input (J/cm2) of a CW 1.5kW CO2 laser on the heat affected zone width (WHAZ) [mm] of a butt-welding of a medium carbon steel. The results are in the file HAZ.csv
1. Suggest a model for the data.
2. Compute the prediction and confidence Heat input equal to 1260 J/cm2

In [ ]:
# Import the necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
import seaborn as sns

# Import the dataset
data = pd.read_csv('../Data/HAZ.csv')

# Inspect the dataset
data.head()

## Point 1. Suggest a model for the data.

In [ ]:
# Plot the data 
plt.plot(data['Heat'],data['HAZ'], 'o')
plt.xlabel('Heat')
plt.ylabel('HAZ')
plt.title('Heat vs HAZ')
plt.grid()
plt.show()

By looking at the scatterplot we can see a clear linear trend between the width (y) and the regressor (HAZ). No information on time order, so we cannot check autocorrelation. 


We fit a linear model to the data. 

In [ ]:
import statsmodels.api as sm
import qdatoolkit as qda

x = data['Heat']
x = sm.add_constant(x) 
y = data['HAZ']
model = sm.OLS(y, x).fit()
qda.summary(model)

In [ ]:
#NORMALITY OF RESIDUALS
residuals = model.resid
fig, axs = plt.subplots(2, 2)
fig.suptitle('Residual Plots')

axs[0,0].set_title('Normal probability plot')
stats.probplot(residuals, dist="norm", plot=axs[0,0])

axs[0,1].set_title('Versus Fits')
axs[0,1].scatter(model.fittedvalues, residuals)

fig.subplots_adjust(hspace=0.5)

axs[1,0].set_title('Histogram')
axs[1,0].hist(residuals)

axs[1,1].set_title('Time series plot')
axs[1,1].plot(np.arange(1, len(residuals)+1), residuals, 'o-')
plt.show()

In [ ]:
_ = qda.Assumptions(residuals).normality()

## Point 2. Compute the prediction and confidence Heat input equal to 1260 J/cm2

In [ ]:
# compute the prediction interval
x_new = 1260
prediction_df = model.get_prediction([1,x_new]).summary_frame(alpha=0.05)
print(prediction_df)

In [ ]:
# get the range of values for the regressor
x_range = np.linspace(data['Heat'].min(), data['Heat'].max(), 100)

# add a constant to the regressor
x_range = sm.add_constant(x_range)

# get the prediction interval for each value of the regressor
prediction_df = model.get_prediction(x_range).summary_frame(alpha=0.05)

# plot the data and the intervals
plt.plot(data['Heat'], data['HAZ'], 'o', color='blue', label='Original data')
plt.plot(x_range[:,1], prediction_df['mean'], '--', color='red', label='Fitted values')
plt.fill_between(x_range[:,1], prediction_df['obs_ci_lower'], prediction_df['obs_ci_upper'], color='green', alpha=0.2)
plt.fill_between(x_range[:,1], prediction_df['mean_ci_lower'], prediction_df['mean_ci_upper'], color='red', alpha=0.2)
plt.title('Confidence and Prediction Intervals')
plt.xlabel('Heat')
plt.ylabel('HAZ')
plt.show()